In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # Notebook 01: Bronze Ingestion
# MAGIC **Food Establishment Inspections -- Chicago & Dallas**
# MAGIC 
# MAGIC This notebook:
# MAGIC 1. Loads raw CSVs from volumes into Bronze Delta tables
# MAGIC 2. Renames columns to clean snake_case (only structural change, no data changes)
# MAGIC 3. Adds audit/metadata columns
# MAGIC 4. Runs profiling for Part 1 deliverable

# COMMAND ----------

# MAGIC %sql
# MAGIC USE CATALOG food_inspections;

In [0]:
from pyspark.sql.functions import (
    lit, current_timestamp, input_file_name, col,
    count, when, countDistinct, sum as spark_sum,
    min as spark_min, max as spark_max
)
from pyspark.sql.types import StructType, StructField, StringType


In [0]:
# MAGIC %md
# MAGIC ## 1. Chicago Bronze Ingestion

# COMMAND ----------

# Chicago schema -- 17 columns, all as StringType in Bronze
# Using exact source column names from the CSV header
chicago_schema = StructType([
    StructField("Inspection ID", StringType(), True),
    StructField("DBA Name", StringType(), True),
    StructField("AKA Name", StringType(), True),
    StructField("License #", StringType(), True),
    StructField("Facility Type", StringType(), True),
    StructField("Risk", StringType(), True),
    StructField("Address", StringType(), True),
    StructField("City", StringType(), True),
    StructField("State", StringType(), True),
    StructField("Zip", StringType(), True),
    StructField("Inspection Date", StringType(), True),
    StructField("Inspection Type", StringType(), True),
    StructField("Results", StringType(), True),
    StructField("Violations", StringType(), True),
    StructField("Latitude", StringType(), True),
    StructField("Longitude", StringType(), True),
    StructField("Location", StringType(), True),
])

df_chicago_raw = (
    spark.read
    .option("header", "true")
    .option("multiLine", "true")
    .option("escape", '"')
    .schema(chicago_schema)
    .csv("/Volumes/food_inspections/raw/chicago_files/")
)

In [0]:
# COMMAND ----------

# Rename to clean snake_case -- this is a structural change only, no data transformation
# We do this in Bronze so every downstream notebook has clean column names
df_chicago_bronze = (
    df_chicago_raw
    .withColumnRenamed("Inspection ID", "inspection_id")
    .withColumnRenamed("DBA Name", "dba_name")
    .withColumnRenamed("AKA Name", "aka_name")
    .withColumnRenamed("License #", "license_num")
    .withColumnRenamed("Facility Type", "facility_type")
    .withColumnRenamed("Risk", "risk")
    .withColumnRenamed("Address", "address")
    .withColumnRenamed("City", "city")
    .withColumnRenamed("State", "state")
    .withColumnRenamed("Zip", "zip")
    .withColumnRenamed("Inspection Date", "inspection_date")
    .withColumnRenamed("Inspection Type", "inspection_type")
    .withColumnRenamed("Results", "results")
    .withColumnRenamed("Violations", "violations")
    .withColumnRenamed("Latitude", "latitude")
    .withColumnRenamed("Longitude", "longitude")
    .withColumnRenamed("Location", "location")
    # Audit columns
    .withColumn("source_city", lit("CHICAGO"))
    .withColumn("load_ts", current_timestamp())
    .withColumn("source_file", input_file_name())
)

In [0]:
# COMMAND ----------

# Write to Bronze
df_chicago_bronze = (
    df_chicago_raw
    .withColumnRenamed("Inspection ID", "inspection_id")
    .withColumnRenamed("DBA Name", "dba_name")
    .withColumnRenamed("AKA Name", "aka_name")
    .withColumnRenamed("License #", "license_num")
    .withColumnRenamed("Facility Type", "facility_type")
    .withColumnRenamed("Risk", "risk")
    .withColumnRenamed("Address", "address")
    .withColumnRenamed("City", "city")
    .withColumnRenamed("State", "state")
    .withColumnRenamed("Zip", "zip")
    .withColumnRenamed("Inspection Date", "inspection_date")
    .withColumnRenamed("Inspection Type", "inspection_type")
    .withColumnRenamed("Results", "results")
    .withColumnRenamed("Violations", "violations")
    .withColumnRenamed("Latitude", "latitude")
    .withColumnRenamed("Longitude", "longitude")
    .withColumnRenamed("Location", "location")
    # Audit columns
    .withColumn("source_city", lit("CHICAGO"))
    .withColumn("load_ts", current_timestamp())
    .withColumn("source_file", col("_metadata.file_path"))
)
(
    df_chicago_bronze
    .write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("food_inspections.bronze.chicago_inspections")
)

chi_count = spark.table("food_inspections.bronze.chicago_inspections").count()
print(f"Chicago Bronze loaded: {chi_count:,} rows, 17 source columns + 3 audit columns")

Chicago Bronze loaded: 308,357 rows, 17 source columns + 3 audit columns


In [0]:
# COMMAND ----------

# Verify schema
spark.table("food_inspections.bronze.chicago_inspections").printSchema()

root
 |-- inspection_id: string (nullable = true)
 |-- dba_name: string (nullable = true)
 |-- aka_name: string (nullable = true)
 |-- license_num: string (nullable = true)
 |-- facility_type: string (nullable = true)
 |-- risk: string (nullable = true)
 |-- address: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- zip: string (nullable = true)
 |-- inspection_date: string (nullable = true)
 |-- inspection_type: string (nullable = true)
 |-- results: string (nullable = true)
 |-- violations: string (nullable = true)
 |-- latitude: string (nullable = true)
 |-- longitude: string (nullable = true)
 |-- location: string (nullable = true)
 |-- source_city: string (nullable = true)
 |-- load_ts: timestamp (nullable = true)
 |-- source_file: string (nullable = true)



In [0]:
# COMMAND ----------

# Preview
display(
    spark.table("food_inspections.bronze.chicago_inspections")
    .select(
        "inspection_id", "dba_name", "aka_name", "license_num",
        "facility_type", "risk", "address", "zip",
        "inspection_date", "inspection_type", "results"
    )
    .limit(10)
)


inspection_id,dba_name,aka_name,license_num,facility_type,risk,address,zip,inspection_date,inspection_type,results
2633568,THE LOBBY KITCHEN,THE LOBBY KITCHEN,3077778,Restaurant,Risk 1 (High),66 E WACKER PL,60601,04/07/2026,License,Fail
2633497,PHO 888,PHO 888,2677617,Restaurant,Risk 1 (High),1137 W ARGYLE ST,60640,04/03/2026,Complaint,Pass w/ Conditions
2633528,FRESH BREW,FRESH BREW,2589819,Restaurant,Risk 2 (Medium),2410 W BRYN MAWR AVE,60659,04/03/2026,Canvass,Out of Business
2633530,TRADER JOE'S STORE #860,TRADER JOE'S,3078248,null,Risk 2 (Medium),6191 N LINCOLN AVE,60659,04/03/2026,License,Not Ready
2633502,O & M QUICK MART,O & M QUICK MART,3073369,Grocery Store,Risk 3 (Low),8625 S STONY ISLAND AVE,60617,04/03/2026,License Re-Inspection,Pass
2633535,SUBWAY,SUBWAY,2846175,Restaurant,Risk 1 (High),1901 W 103RD ST,60643,04/03/2026,Canvass,Fail
2633486,HABEEB MEAT STOP,HABEEB MEAT STOP,3074015,Grocery Store,Risk 2 (Medium),2238-2240 W DEVON AVE,60659,04/03/2026,License Re-Inspection,Pass
2633512,PD COFFEE SHOP,PD COFFEE SHOP,3069421,Restaurant,Risk 2 (Medium),2410 W BRYN MAWR AVE,60659,04/03/2026,License,Pass w/ Conditions
2633488,EASY STREET PIZZA,EAZY STREET PIZZA,2517172,Restaurant,Risk 1 (High),3750-3754 N CENTRAL AVE,60634,04/03/2026,Short Form Complaint,Pass
2633533,IZAKAYA TOKYO,IZAKAYA TOKYO,3078062,Restaurant,Risk 1 (High),3335-3337 N HALSTED ST,60657,04/03/2026,License,Pass w/ Conditions


In [0]:
# COMMAND ----------

# MAGIC %md
# MAGIC ## 2. Dallas Bronze Ingestion

# COMMAND ----------

# Dallas: 114 columns. Let Spark infer column names from header.
# All values kept as StringType (no casting in Bronze).
df_dallas_raw = (
    spark.read
    .option("header", "true")
    .option("multiLine", "true")
    .option("escape", '"')
    .option("inferSchema", "false")
    .csv("/Volumes/food_inspections/raw/dallas_files/")
)


In [0]:
# COMMAND ----------

# Rename Dallas columns to clean snake_case
# Core columns renamed explicitly
# Violation columns renamed programmatically

# Start with core columns
# Rename Dallas columns to clean snake_case
# Rename Dallas columns to clean snake_case
df_dallas_bronze = (
    df_dallas_raw
    .withColumnRenamed("Restaurant Name", "restaurant_name")
    .withColumnRenamed("Inspection Type", "inspection_type")
    .withColumnRenamed("Inspection Date", "inspection_date")
    .withColumnRenamed("Inspection Score", "inspection_score")
    .withColumnRenamed("Street Number", "street_number")
    .withColumnRenamed("Street Name", "street_name")
    .withColumnRenamed("Street Direction", "street_direction")
    .withColumnRenamed("Street Type", "street_type")
    .withColumnRenamed("Street Unit", "street_unit")
    .withColumnRenamed("Street Address", "street_address")
    .withColumnRenamed("Zip Code", "zip_code")
    .withColumnRenamed("Inspection Month", "inspection_month")
    .withColumnRenamed("Inspection Year", "inspection_year")
    .withColumnRenamed("Lat Long Location", "lat_long_location")
)

# Rename violation columns programmatically (25 groups of 4)
for i in range(1, 26):
    df_dallas_bronze = (
        df_dallas_bronze
        .withColumnRenamed(f"Violation Description - {i}", f"violation_description_{i}")
        .withColumnRenamed(f"Violation Points - {i}", f"violation_points_{i}")
        .withColumnRenamed(f"Violation Detail - {i}", f"violation_detail_{i}")
        .withColumnRenamed(f"Violation Memo - {i}", f"violation_memo_{i}")
    )

# Handle potential double-space typo (safe -- does nothing if column doesn't exist)
df_dallas_bronze = df_dallas_bronze.withColumnRenamed("Violation  Memo - 20", "violation_memo_20")

# Add audit columns
df_dallas_bronze = (
    df_dallas_bronze
    .withColumn("source_city", lit("DALLAS"))
    .withColumn("load_ts", current_timestamp())
    .withColumn("source_file", col("_metadata.file_path"))
)


In [0]:
# COMMAND ----------

# Write to Bronze
(
    df_dallas_bronze
    .write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("food_inspections.bronze.dallas_inspections")
)

dal_count = spark.table("food_inspections.bronze.dallas_inspections").count()
print(f"Dallas Bronze loaded: {dal_count:,} rows, 114 source columns + 3 audit columns")


Dallas Bronze loaded: 78,984 rows, 114 source columns + 3 audit columns


In [0]:
# COMMAND ----------
# Verify no columns with spaces remain in Dallas
dal_cols = spark.table("food_inspections.bronze.dallas_inspections").columns
dirty_cols = [c for c in dal_cols if " " in c]
if dirty_cols:
    print(f"WARNING: {len(dirty_cols)} columns still have spaces:")
    for c in dirty_cols:
        print(f"  '{c}'")
else:
    print(f"All {len(dal_cols)} columns are clean snake_case")

print(f"\nTotal columns: {len(dal_cols)}")

All 117 columns are clean snake_case

Total columns: 117


In [0]:
# COMMAND ----------

# Preview core Dallas columns
display(
    spark.table("food_inspections.bronze.dallas_inspections")
    .select(
        "restaurant_name", "inspection_type", "inspection_date",
        "inspection_score", "street_address", "zip_code",
        "inspection_month", "inspection_year", "lat_long_location"
    )
    .limit(10)
)


restaurant_name,inspection_type,inspection_date,inspection_score,street_address,zip_code,inspection_month,inspection_year,lat_long_location
TIENDA Y RESTAURANTE LA CAMPIñA SALvadorena,Routine,10/03/2016,66,10818 DENNIS RD 102,75229,Oct 2016,FY2017,"(32.895847, -96.881391)"
TORTILLERIA LA ESPIGA,Routine,10/03/2016,82,1328 N JIM MILLER RD #104,75217,Oct 2016,FY2017,"(32.73556, -96.700079)"
TMGM @ KEETON PARK GOLF COURSE,Routine,10/03/2016,80,2323 N JIM MILLER RD,75227,Oct 2016,FY2017,"(32.756246, -96.701964)"
TORTAS Y TACOS EL RANCHITO #4,Routine,10/03/2016,83,5560 E GRAND AVE,75223,Oct 2016,FY2017,"(32.793596, -96.747508)"
ROYAL TAQUERIA,Routine,10/03/2016,81,10818 DENNIS RD #103,75229,Oct 2016,FY2017,"(32.895847, -96.881391)"
STARBUCKS COFFEE #6229,Routine,10/03/2016,89,5350 W LOVERS LN #125,75209,Oct 2016,FY2017,"(32.851137, -96.82012)"
WILLIE'S,Routine,10/03/2016,92,1105 S BEACON ST,75223,Oct 2016,FY2017,"(32.794517, -96.748303)"
MAPLE LEAF DINER,Routine,10/03/2016,95,12817 PRESTON RD #129,75230,Oct 2016,FY2017,"(32.92346, -96.80363)"
HENDERSON CHICKEN,Routine,10/03/2016,80,1328 N JIM MILLER RD #108,75217,Oct 2016,FY2017,"(32.73556, -96.700079)"
ROYAL DOLLAR & FOOD STORE,Routine,10/03/2016,80,10818 DENNIS RD #103,75229,Oct 2016,FY2017,"(32.895847, -96.881391)"


In [0]:
# COMMAND ----------

# Preview first violation group
display(
    spark.table("food_inspections.bronze.dallas_inspections")
    .select(
        "restaurant_name",
        "violation_description_1", "violation_points_1",
        "violation_detail_1", "violation_memo_1"
    )
    .filter("violation_description_1 IS NOT NULL")
    .limit(5)
)


restaurant_name,violation_description_1,violation_points_1,violation_detail_1,violation_memo_1
TIENDA Y RESTAURANTE LA CAMPIñA SALvadorena,"*01 Cooling -- within 2 hours, 135-70øF",3,"228.75 Food. Time and temperature control. (d) Cooling. (1) Cooked temperature/time controlled for safety food shall be cooled: (A) within two hours, from 57 degrees Celsius (135 degrees Fahrenheit) to 21 degrees C (70 degrees Fahrenheit); and",null
TORTILLERIA LA ESPIGA,"*39 In-use utensils, between-use storage. During pauses in food preparation or dispensing, food prep",1,"228.68 Food. Preventing Contamination From Equipment, Utensils, and Linens. (b) In-use utensils, between-use storage. During pauses in food preparation or dispensing, food preparation and dispensing utensils shall be stored: (2) in food that is not time/temperature controlled for safety (TCS) with their handles above the top of the food within containers or equipment that can be closed, such as bins of sugar, flour, or cinnamon;",cannot use water containers to keep utensils to reuse again
TMGM @ KEETON PARK GOLF COURSE,*12 A food employee shall comply with an exclusion or RESTRICTION,3,"228.35 Management and Personnel. Responsibilities and Reporting Symptoms and Diagnosis. (f) A food employee shall: (2) comply with a restriction as specified in õ228.36(4)(B), (5)(B), (6)(B), (7), (8)(B), or õ228.36 (8), (9), or (10) and comply with the provisions specified in õ228.37(4) - (10) of this title.",missing employee health policies
TORTAS Y TACOS EL RANCHITO #4,*03 Food products not maintained at 135øF or above,3,"228.75 Food. Time and temperature control. (f) Time/temperature controlled for safety food, hot and cold holding. (1) Except during preparation, cooking, or cooling, or when time is used as the public health control as specified in subsection (i) of this sectionin paragraphs (2) and (3) of this subsection, TCS food shall be maintained: (A) at 57 degrees Celsius (135 degrees Fahrenheit) or above, except that roasts cooked to a temperature and for a time specified in õ228.71(a)(2) of this title or reheated as specified in subsection õ228.73(e) of this title may be held at a temperature of 54 degrees Celsius (130 degrees Fahrenheit);",refried beans were at 130 degrees in some area and 125 in other areas ( stir product while on the hot holding unit.)
ROYAL TAQUERIA,"*31 Individual, disposable towels",2,"228.175 Physical Facilities. Handwashing Sinks. (c) Hand drying provision. Each handwashing lavatory or group of adjacent lavatories shall be provided with: (1) individual, disposable towels;",null


In [0]:
# COMMAND ----------

# MAGIC %md
# MAGIC ## 3. Row Count Summary

# COMMAND ----------

# MAGIC %sql
# MAGIC SELECT 'Chicago' AS source, COUNT(*) AS row_count
# MAGIC FROM food_inspections.bronze.chicago_inspections
# MAGIC UNION ALL
# MAGIC SELECT 'Dallas' AS source, COUNT(*) AS row_count
# MAGIC FROM food_inspections.bronze.dallas_inspections;

# COMMAND ----------

# MAGIC %md
# MAGIC ## 4. Chicago Profiling

# COMMAND ----------

# MAGIC %md
# MAGIC ### 4a. Null / Empty Analysis

# COMMAND ----------

df_chi = spark.table("food_inspections.bronze.chicago_inspections")
chi_total = df_chi.count()

chi_source_cols = [
    "inspection_id", "dba_name", "aka_name", "license_num",
    "facility_type", "risk", "address", "city", "state", "zip",
    "inspection_date", "inspection_type", "results",
    "violations", "latitude", "longitude", "location"
]

# Count nulls and empty strings per column
null_results = {}
for c in chi_source_cols:
    null_count = df_chi.filter(
        col(c).isNull() | (col(c) == "")
    ).count()
    null_results[c] = null_count

print(f"CHICAGO PROFILING: {chi_total:,} total rows\n")
print(f"{'Column':<20} {'Nulls/Empty':>12} {'%':>7}  {'Action'}")
print("-" * 70)
for c in chi_source_cols:
    n = null_results[c]
    pct = (n / chi_total) * 100 if chi_total > 0 else 0
    if c in ("dba_name", "inspection_date", "inspection_type", "zip", "results"):
        action = "DROP ROW (validation rule)" if pct > 0 else "OK"
    elif pct > 50:
        action = "HIGH NULL RATE"
    else:
        action = ""
    print(f"{c:<20} {n:>12,} {pct:>6.1f}%  {action}")


CHICAGO PROFILING: 308,357 total rows

Column                Nulls/Empty       %  Action
----------------------------------------------------------------------
inspection_id                   0    0.0%  
dba_name                        0    0.0%  OK
aka_name                    2,420    0.8%  
license_num                    19    0.0%  
facility_type               5,323    1.7%  
risk                           88    0.0%  
address                         0    0.0%  
city                          180    0.1%  
state                          67    0.0%  
zip                            42    0.0%  DROP ROW (validation rule)
inspection_date                 0    0.0%  OK
inspection_type                 1    0.0%  DROP ROW (validation rule)
results                         0    0.0%  OK
violations                 86,472   28.0%  
latitude                    1,048    0.3%  
longitude                   1,048    0.3%  
location                    1,048    0.3%  


In [0]:
# COMMAND ----------

# MAGIC %md
# MAGIC ### 4b. Distinct Values (Cardinality)

# COMMAND ----------

distinct_data = {}
for c in chi_source_cols:
    distinct_data[c] = df_chi.select(countDistinct(col(c))).collect()[0][0]

print(f"{'Column':<20} {'Distinct Values':>16}")
print("-" * 38)
for c in chi_source_cols:
    print(f"{c:<20} {distinct_data[c]:>16,}")


Column                Distinct Values
--------------------------------------
inspection_id                 308,357
dba_name                       34,620
aka_name                       32,945
license_num                    48,293
facility_type                     520
risk                                4
address                        33,061
city                               89
state                               6
zip                               132
inspection_date                 4,094
inspection_type                   110
results                             7
violations                    220,265
latitude                       18,812
longitude                      18,812
location                       18,812


In [0]:
%sql
SELECT
    results,
    COUNT(*) AS cnt,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 1) AS pct
FROM food_inspections.bronze.chicago_inspections
GROUP BY results
ORDER BY cnt DESC;

results,cnt,pct
Pass,159411,51.7
Fail,59453,19.3
Pass w/ Conditions,45929,14.9
Out of Business,25469,8.3
No Entry,13669,4.4
Not Ready,4332,1.4
Business Not Located,94,0.0


In [0]:
%sql
SELECT
    inspection_type,
    COUNT(*) AS cnt,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 1) AS pct
FROM food_inspections.bronze.chicago_inspections
GROUP BY inspection_type
ORDER BY cnt DESC;

inspection_type,cnt,pct
Canvass,160017,51.9
License,41103,13.3
Canvass Re-Inspection,34371,11.1
Complaint,28939,9.4
License Re-Inspection,12592,4.1
Complaint Re-Inspection,12067,3.9
Short Form Complaint,9072,2.9
Non-Inspection,5327,1.7
Suspected Food Poisoning,1000,0.3
Consultation,679,0.2


In [0]:
%sql
SELECT
    risk,
    COUNT(*) AS cnt,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 1) AS pct
FROM food_inspections.bronze.chicago_inspections
GROUP BY risk
ORDER BY cnt DESC;

risk,cnt,pct
Risk 1 (High),228825,74.2
Risk 2 (Medium),55323,17.9
Risk 3 (Low),24034,7.8
null,88,0.0
All,87,0.0


In [0]:
%sql
SELECT
    facility_type,
    COUNT(*) AS cnt,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 1) AS pct
FROM food_inspections.bronze.chicago_inspections
GROUP BY facility_type
ORDER BY cnt DESC
LIMIT 20;

facility_type,cnt,pct
Restaurant,209072,67.8
Grocery Store,37131,12.0
School,19258,6.2
Children's Services Facility,7432,2.4
null,5323,1.7
Bakery,4461,1.4
Daycare Above and Under 2 Years,4081,1.3
Daycare (2 - 6 Years),3247,1.1
Long Term Care,2377,0.8
Catering,1901,0.6


In [0]:
%sql
SELECT
    MIN(inspection_date) AS earliest_date,
    MAX(inspection_date) AS latest_date,
    COUNT(DISTINCT zip) AS distinct_zips,
    SUM(CASE WHEN zip NOT RLIKE '^[0-9]{5}$' THEN 1 ELSE 0 END) AS invalid_zips,
    SUM(CASE WHEN violations IS NULL OR violations = '' THEN 1 ELSE 0 END) AS no_violations_count
FROM food_inspections.bronze.chicago_inspections;

earliest_date,latest_date,distinct_zips,invalid_zips,no_violations_count
01/02/2013,12/31/2025,132,0,86472


In [0]:
%sql
SELECT
    SUM(CASE WHEN violations LIKE '%&amp;%' THEN 1 ELSE 0 END) AS has_amp_entity,
    SUM(CASE WHEN violations LIKE '%&#39;%' THEN 1 ELSE 0 END) AS has_apostrophe_entity,
    SUM(CASE WHEN violations LIKE '%&quot;%' THEN 1 ELSE 0 END) AS has_quote_entity,
    COUNT(*) AS total_with_violations
FROM food_inspections.bronze.chicago_inspections
WHERE violations IS NOT NULL AND violations != '';

has_amp_entity,has_apostrophe_entity,has_quote_entity,total_with_violations
0,0,0,221885


In [0]:
%sql
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT inspection_id) AS distinct_ids,
    COUNT(*) - COUNT(DISTINCT inspection_id) AS duplicate_rows
FROM food_inspections.bronze.chicago_inspections;

total_rows,distinct_ids,duplicate_rows
308357,308357,0


In [0]:
%sql
SELECT
    SUM(CASE WHEN UPPER(violations) LIKE '%PRIORITY VIOLATION%' THEN 1 ELSE 0 END) AS has_priority_violation,
    SUM(CASE WHEN UPPER(violations) LIKE '%PRIORITY FOUNDATION VIOLATION%' THEN 1 ELSE 0 END) AS has_priority_foundation,
    SUM(CASE WHEN UPPER(violations) LIKE '%CITATION ISSUED%' THEN 1 ELSE 0 END) AS has_citation_issued,
    SUM(CASE WHEN UPPER(violations) LIKE '%NO CITATION ISSUED%' THEN 1 ELSE 0 END) AS has_no_citation,
    SUM(CASE WHEN UPPER(violations) LIKE '%CRITICAL%' THEN 1 ELSE 0 END) AS has_word_critical,
    SUM(CASE WHEN UPPER(violations) LIKE '%URGENT%' THEN 1 ELSE 0 END) AS has_word_urgent,
    COUNT(*) AS total_with_violations
FROM food_inspections.bronze.chicago_inspections
WHERE violations IS NOT NULL AND violations != '';

has_priority_violation,has_priority_foundation,has_citation_issued,has_no_citation,has_word_critical,has_word_urgent,total_with_violations
14557,30720,55586,21189,17541,0,221885


In [0]:
# COMMAND ----------

df_dal = spark.table("food_inspections.bronze.dallas_inspections")
dal_total = df_dal.count()

dal_core_cols = [
    "restaurant_name", "inspection_type", "inspection_date",
    "inspection_score", "street_number", "street_name",
    "street_direction", "street_type", "street_unit",
    "street_address", "zip_code",
    "inspection_month", "inspection_year", "lat_long_location"
]

null_results_dal = {}
for c in dal_core_cols:
    null_count = df_dal.filter(
        col(c).isNull() | (col(c) == "")
    ).count()
    null_results_dal[c] = null_count

print(f"DALLAS PROFILING: {dal_total:,} total rows\n")
print(f"{'Column':<22} {'Nulls/Empty':>12} {'%':>7}  {'Action'}")
print("-" * 65)
for c in dal_core_cols:
    n = null_results_dal[c]
    pct = (n / dal_total) * 100 if dal_total > 0 else 0
    if c in ("restaurant_name", "inspection_date", "inspection_type", "zip_code"):
        action = "DROP ROW (validation rule)" if pct > 0 else "OK"
    elif pct > 50:
        action = "HIGH NULL RATE"
    else:
        action = ""
    print(f"{c:<22} {n:>12,} {pct:>6.1f}%  {action}")

DALLAS PROFILING: 78,984 total rows

Column                  Nulls/Empty       %  Action
-----------------------------------------------------------------
restaurant_name                  11    0.0%  DROP ROW (validation rule)
inspection_type                   0    0.0%  OK
inspection_date                   0    0.0%  OK
inspection_score                  0    0.0%  
street_number                     0    0.0%  
street_name                       0    0.0%  
street_direction             52,898   67.0%  HIGH NULL RATE
street_type                   1,674    2.1%  
street_unit                  50,804   64.3%  HIGH NULL RATE
street_address                    0    0.0%  
zip_code                          0    0.0%  OK
inspection_month                  0    0.0%  
inspection_year                   0    0.0%  
lat_long_location             8,638   10.9%  


In [0]:
%sql
SELECT
    MIN(CAST(inspection_score AS INT)) AS min_score,
    MAX(CAST(inspection_score AS INT)) AS max_score,
    ROUND(AVG(CAST(inspection_score AS INT)), 1) AS avg_score,
    SUM(CASE WHEN CAST(inspection_score AS INT) > 100 THEN 1 ELSE 0 END) AS scores_over_100,
    SUM(CASE WHEN inspection_score IS NULL OR inspection_score = '' THEN 1 ELSE 0 END) AS null_scores
FROM food_inspections.bronze.dallas_inspections;

min_score,max_score,avg_score,scores_over_100,null_scores
-26,100,90.9,0,0


In [0]:
%sql
SELECT
    inspection_type,
    COUNT(*) AS cnt,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 1) AS pct
FROM food_inspections.bronze.dallas_inspections
GROUP BY inspection_type
ORDER BY cnt DESC;

inspection_type,cnt,pct
Routine,78019,98.8
Follow-up,935,1.2
Complaint,30,0.0


In [0]:
from pyspark.sql.functions import when as spark_when

viol_desc_cols = [f"violation_description_{i}" for i in range(1, 26)]

count_expr = sum([
    spark_when(col(c).isNotNull() & (col(c) != ""), 1).otherwise(0)
    for c in viol_desc_cols
])

df_viol_dist = (
    df_dal
    .withColumn("violation_count", count_expr)
    .groupBy("violation_count")
    .count()
    .orderBy("violation_count")
)

display(df_viol_dist)

violation_count,count
0,6579
1,7550
2,8941
3,8884
4,8224
5,7415
6,6562
7,5667
8,4827
9,4068


In [0]:
%sql
SELECT
    SUM(CASE WHEN lat_long_location LIKE '%object Object%' THEN 1 ELSE 0 END) AS js_garbage,
    SUM(CASE WHEN lat_long_location LIKE '%°%' THEN 1 ELSE 0 END) AS has_degree_symbol,
    SUM(CASE WHEN lat_long_location IS NULL OR lat_long_location = '' THEN 1 ELSE 0 END) AS null_coords,
    COUNT(*) AS total
FROM food_inspections.bronze.dallas_inspections;

js_garbage,has_degree_symbol,null_coords,total
0,0,8638,78984


In [0]:
%sql
SELECT DISTINCT violation_points_1
FROM food_inspections.bronze.dallas_inspections
WHERE violation_points_1 IS NOT NULL AND violation_points_1 != ''
ORDER BY violation_points_1;

violation_points_1
1
2
3
4


In [0]:
%sql
SELECT restaurant_name, inspection_date, inspection_score, street_address
FROM food_inspections.bronze.dallas_inspections
WHERE CAST(inspection_score AS INT) < 0
LIMIT 10;

restaurant_name,inspection_date,inspection_score,street_address
MI PUEBLITO TAQUERIA Y PA,06/18/2023,-26,2222 GUS THOMASSON RD
KFC,06/23/2023,-13,106 W ILLINOIS AVE
THE LAST STAND,09/13/2023,-15,330 W DAVIS ST #120
MACARON,09/15/2023,-3,211 S AKARD ST #108
2AT&T FOOD HALL - COMMON AREA (LVL 01),09/15/2023,-5,211 S AKARD ST #100
NOODLES,09/15/2023,-5,211 S AKARD ST #112


In [0]:
from pyspark.sql.functions import col, when as spark_when, lit, current_timestamp

df_dal = spark.table("food_inspections.bronze.dallas_inspections")
dal_total = df_dal.count()

viol_desc_cols = [f"violation_description_{i}" for i in range(1, 26)]

count_expr_2 = sum([
    spark_when(col(c).isNotNull() & (col(c) != ""), 1).otherwise(0)
    for c in viol_desc_cols
])

fail_count = (
    df_dal
    .withColumn("violation_count", count_expr_2)
    .filter(
        (col("inspection_score").cast("int") >= 90) &
        (col("violation_count") > 3)
    )
    .count()
)

print(f"Dallas rows failing 'score >= 90 with > 3 violations' rule: {fail_count:,}")
print(f"({(fail_count / dal_total * 100):.2f}% of total)")

Dallas rows failing 'score >= 90 with > 3 violations' rule: 18,301
(23.17% of total)


In [0]:
%sql
SELECT
    MIN(inspection_date) AS earliest_date,
    MAX(inspection_date) AS latest_date,
    COUNT(DISTINCT zip_code) AS distinct_zips,
    SUM(CASE WHEN zip_code NOT RLIKE '^[0-9]{5}$' THEN 1 ELSE 0 END) AS invalid_zips,
    SUM(CASE WHEN zip_code IS NULL OR zip_code = '' THEN 1 ELSE 0 END) AS null_zips
FROM food_inspections.bronze.dallas_inspections;

earliest_date,latest_date,distinct_zips,invalid_zips,null_zips
01/01/2018,12/31/2022,156,259,0


In [0]:
%sql
SELECT
    SUM(CASE WHEN lat_long_location LIKE '%object Object%' THEN 1 ELSE 0 END) AS js_garbage,
    SUM(CASE WHEN lat_long_location LIKE '%°%' THEN 1 ELSE 0 END) AS has_degree_symbol,
    SUM(CASE WHEN lat_long_location IS NULL OR lat_long_location = '' THEN 1 ELSE 0 END) AS null_coords,
    COUNT(*) AS total
FROM food_inspections.bronze.dallas_inspections;

js_garbage,has_degree_symbol,null_coords,total
0,0,8638,78984


In [0]:
%sql
SELECT
    violation_points_1 AS points_value,
    COUNT(*) AS cnt
FROM food_inspections.bronze.dallas_inspections
WHERE violation_points_1 IS NOT NULL AND violation_points_1 != ''
GROUP BY violation_points_1
ORDER BY cnt DESC;

points_value,cnt
3,28550
1,24947
2,18907
4,1


In [0]:
mapping = [
    ("Business Name",      "dba_name",           "restaurant_name",       "Direct map, standardize casing"),
    ("Also Known As",      "aka_name",            "N/A",                   "Chicago only"),
    ("License/ID",         "license_num",         "N/A (derive NK)",       "Chicago has license; Dallas derive from name+addr+zip"),
    ("Facility Type",      "facility_type",       "N/A",                   "Chicago only"),
    ("Risk Level",         "risk",                "N/A",                   "Chicago only"),
    ("Address",            "address",             "street_address",        "Direct map"),
    ("City",               "city",                "Literal 'DALLAS'",      "Hardcode for Dallas"),
    ("State",              "state",               "Literal 'TX'",          "Hardcode for Dallas"),
    ("Zip Code",           "zip",                 "zip_code",              "Both have zip"),
    ("Inspection Date",    "inspection_date",     "inspection_date",       "Both MM/dd/yyyy, verify"),
    ("Inspection Type",    "inspection_type",     "inspection_type",       "Different systems (Canvass vs Routine)"),
    ("Result",             "results",             "Derive from score",     "Chicago text; Dallas derive from score"),
    ("Score",              "Derive from results", "inspection_score",      "Chicago derive; Dallas has numeric"),
    ("Violations",         "violations (blob)",   "violation_desc 1-25",   "Chicago parse; Dallas unpivot"),
    ("Latitude",           "latitude",            "lat_long_location",     "Chicago direct; Dallas parse"),
    ("Longitude",          "longitude",           "lat_long_location",     "Chicago direct; Dallas parse"),
    ("Inspection Month",   "N/A",                 "inspection_month",      "Dallas only"),
    ("Inspection Year",    "N/A",                 "inspection_year",       "Dallas only"),
]

print(f"{'Target Concept':<20} {'Chicago Source':<22} {'Dallas Source':<22} {'Notes'}")
print("=" * 100)
for concept, chi, dal, notes in mapping:
    print(f"{concept:<20} {chi:<22} {dal:<22} {notes}")

Target Concept       Chicago Source         Dallas Source          Notes
Business Name        dba_name               restaurant_name        Direct map, standardize casing
Also Known As        aka_name               N/A                    Chicago only
License/ID           license_num            N/A (derive NK)        Chicago has license; Dallas derive from name+addr+zip
Facility Type        facility_type          N/A                    Chicago only
Risk Level           risk                   N/A                    Chicago only
Address              address                street_address         Direct map
City                 city                   Literal 'DALLAS'       Hardcode for Dallas
State                state                  Literal 'TX'           Hardcode for Dallas
Zip Code             zip                    zip_code               Both have zip
Inspection Date      inspection_date        inspection_date        Both MM/dd/yyyy, verify
Inspection Type      inspection_type        

## Profiling Summary

### Chicago: 308,357 rows | 17 columns | 

- **0 duplicate inspection IDs** -- every row is unique
- **0 invalid zips** -- all match 5-digit format
- **0 HTML entities** -- CSV export already decoded
- **86,472 rows (28%) have no violations** -- will be dropped per validation rules
- **Null rates:** facility_type 5,323 (1.7%), aka_name 2,420 (0.8%), latitude/longitude 1,048 (0.3%), zip 42, inspection_type 1
- **7 result types:** Pass (51.7%), Fail (19.3%), Pass w/ Conditions (14.9%), Out of Business (8.3%), No Entry (4.4%), Not Ready (1.4%), Business Not Located (0.0%)
- **110 inspection types** with heavy casing inconsistency (CANVASS vs Canvass vs canvass). Top 8 cover 98.6% of data.
- **Priority keywords in violations:** PRIORITY VIOLATION (14,557), PRIORITY FOUNDATION VIOLATION (30,720), CITATION ISSUED (55,586), word CRITICAL (17,541), word URGENT (0)

### Dallas: 78,984 rows | 114 columns | Date range: 01/01/2018 - 12/31/2022

- **No restaurant ID in source** -- must derive natural key from name + address + zip
- **11 null restaurant names** -- will be dropped
- **259 invalid zips** -- not 5-digit format, will be dropped
- **6 negative scores** (min: -26) -- data entry errors, will be dropped
- **0 scores over 100**
- **6,579 rows (8.3%) have zero violations** -- will be dropped
- **18,301 rows (23.2%) fail score >= 90 with > 3 violations rule** -- will be dropped
- **8,638 null coordinates** (10.9%), 0 JS garbage, 0 degree symbols
- **Violation points values:** 1, 2, 3, and one row with 4
- **3 inspection types:** Routine (98.8%), Follow-up (1.2%), Complaint (0.0%)

### Estimated Silver Drops

| Rule | Chicago | Dallas |
|---|---|---|
| Null restaurant name | 0 | 11 |
| Null inspection type | 1 | 0 |
| Invalid/null zip | 42 | 259 |
| Zero violations | 86,472 | 6,579 |
| Negative score | N/A | 6 |
| Score >= 90 with > 3 violations | N/A | 18,301 |
| **Estimated total** | **~86,515** | **~25,156** |

### Common Attributes Mapping

- **Both have:** restaurant name, address, zip, inspection date, inspection type, violations
- **Chicago-only:** license number, facility type, risk level, AKA name, inspection ID
- **Dallas-only:** numeric score, inspection month/year, violation severity points, address components
